### 복습
- answers, summarys 데이터에서 상위의 100개의 데이터를 train 사용
- 하위의 20개의 데이터를 validation으로 사용
- DataDict를 생성
- 모델은 'digit82/kobert-summarization' 사용
- tokenizer를 이용해서 토큰화 인코딩
- kobart모델 trainer를 이용하여 반복 학습
    - input의 최대 사이즈 : 512
    - output의 최대 사이즈 : 256

In [4]:
import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
from konlpy.tag import Komoran

In [12]:
model_name='digit82/kobart-summarization'
#AutoTokenizer가 아닌 Komoran 객체를 생성하는 이유는?
    #실제값(요약본)과 예측값(텍스트 생성) 단어들의 일치도를 확인하기 위함
komoran=Komoran()

In [6]:
df=pd.read_csv('인터뷰.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12799 entries, 0 to 12798
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   answer   12799 non-null  str  
 1   summary  12799 non-null  str  
dtypes: str(2)
memory usage: 17.0 MB


In [7]:
answers=df['answer'].tolist()
summarys=df['summary'].tolist()

In [8]:
train_docs=answers[:100]
train_sums=summarys[:100]

valid_docs=answers[-20:]
valid_sums=summarys[-20:]

In [9]:
#Dataset으로 이루어진 DatasetDict
raw_ds=DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document': train_docs,
                'summary' : train_sums
            }
        ),
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs,
                'summary' : valid_sums
            }
        )
    }
)
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 20
    })
})

In [10]:
DatasetDict(
    {
        'train':Dataset.from_pandas(df.iloc[:100]),
        'validation':Dataset.from_pandas(df.iloc[-20:])
    }
)

DatasetDict({
    train: Dataset({
        features: ['answer', 'summary'],
        num_rows: 100
    })
    validation: Dataset({
        features: ['answer', 'summary'],
        num_rows: 20
    })
})

In [13]:
tokenizer=AutoTokenizer.from_pretrained(model_name,use_fast=True)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)
#you passed ... : 경고 메시지 (설정 충돌)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--digit82--kobart-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 3864.34it/s]


In [15]:
#입출력 길이 생성
max_input_len=512
max_target_len=256

In [20]:
#tokenizer 함수
def token_fn(batch):
    #document의 토큰화
    inputs=tokenizer(
        batch['document'],
        max_length=max_input_len,
        padding="max_length",                   #패딩 토큰으로 채워서 길이 유지
        truncation=True                         #max_length보다 텍스트가 길다면 최대 길이 외의 데이터를 제거
    )

    # with tokenizer.as_target_tokenizer():           #임시 토크나이저 생성
    labels=tokenizer(
        batch['summary'],
        max_length=max_target_len,
        padding="max_length",
        truncation=True
    )

    labels_ids=np.array(labels['input_ids'])
    #패딩 토큰의 값들을 -100으로 변환 (손실 함수에서 손실 값을 계산 안 하는 기본값 설정)
    labels_ids[labels_ids == tokenizer.pad_token_id] == -100
    inputs['labels']=labels_ids.tolist()

    return inputs

In [21]:
tokenized_ds=raw_ds.map(
    token_fn, batched=True,remove_columns=['document','summary']
)

Map: 100%|██████████| 20/20 [00:00<00:00, 106.41 examples/s]


In [22]:
data_collator=DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
    )

In [23]:
rouge=evaluate.load('rouge')

In [24]:
#검증 함수
def metrics(eval_pred):
    preds,labels=eval_pred

    #labels에서 패딩 토큰의 값들을 다시 tokenizer의 
    labels=np.where(
        labels!=100, labels, tokenizer.pad_token_id
    )
    #텍스트 디코딩 (단어 사전의 id에 대응하는 문자로 변환)
    pred_str=tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str=tokenizer.batch_decode(labels, skip_special_tokens=True)

    #디코딩 과정에서 좌우의 공백이 존재하는 경우에는 측정 값이 달라질 수 있으므로 공백을 재거
    pred_str=[doc.strip() for doc in pred_str]
    label_str=[doc.strip() for doc in label_str]

    #rouge 계산
    result=rouge.compute(
        predictions=pred_str,
        references=label_str,
        tokenizer=lambda x : komoran.morphs(x)
    )

    #rouge를 보기 편한 형태로 변환
    result = { k : round(v*100, 2) for k, v in result.items()}

    return result

In [25]:
args=Seq2SeqTrainingArguments(
    output_dir='./kobert',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-05,
    num_train_epochs=5,
    logging_steps=1,

    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,

    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    dataloader_num_workers=3
)

In [28]:
trainer=Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=metrics
)

In [44]:
#예측 : answers의 임의의 데이터를 선택 -> 요약본 예측 텍스트 생성
test_text=answers[200]
# test_text=answers
# test_text

In [45]:
inputs=tokenizer(
    test_text,
    return_tensors='pt',
    truncation=True,
    max_length=max_input_len,
    padding='max_length'
)

In [ ]:
inputs

In [46]:
gen_ids=model.generate(
    **inputs.to(model.device),
    max_new_tokens=500,
    min_new_tokens=10,
    num_beams=4,
    do_sample=True,
    length_penalty=0.6,
    no_repeat_ngram_size=2,
    repetition_penalty=3.0,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

In [47]:
gen_ids

tensor([[    0, 24407, 18964, 22424, 14385, 20658, 12926, 14174, 11440, 17150,
         17216, 15350,  9049, 14464, 16750, 15995, 15416, 14160, 14108, 21548,
         17053, 17131, 17939, 21194, 16145, 14339, 16707, 14056, 15706, 12264,
         28326, 17150, 20244, 15940, 14955, 14339, 14721, 15382, 14030, 14060,
         28885, 16701, 16494, 16960, 14207, 14032, 14082, 14376, 20677, 14882,
         14838, 24135, 27661,     1]])

In [48]:
print(tokenizer.decode(gen_ids, skip_special_tokens=True))

['백지연이 지은 크리티컬 매스 라는 책을 읽게 된 이유는 제가 정말 남들이 감동시킬 만한 노력을 한다면 그것은 타인들이 다 알아줄 것이다 라는 이야기가 있고, 그래서 타인의 이야기에서 제 장점을 끌어내고 그것을 할 수 있는 그런 깨달음을 얻었기 때문이다.']


In [49]:
summarys[2000]

'저는 대학 생활을 하면서 리더십이 부족하다고 느껴져서 교직 이수를 신청했습니다. 학교 현장 실습 동안 수많은 학생들 앞에서 가르쳐야 해서 용기도 생겼고, 수업 준비를 철저히 했습니다. 또한 양질의 학습 내용을 전달하기 위해 공부하고, 학생 개개인의 학습 목표를 성취할 수 있도록 책임 의식을 가졌습니다.'

In [51]:
model2=AutoModelForSeq2SeqLM.from_pretrained('./kobart_exam')

Loading weights: 100%|██████████| 260/260 [00:01<00:00, 150.82it/s]


In [52]:
gen_ids=model2.generate(
    **inputs.to(model.device),
    max_new_tokens=500,
    min_new_tokens=10,
    num_beams=4,
    do_sample=True,
    length_penalty=0.6,
    no_repeat_ngram_size=2,
    repetition_penalty=3.0,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)
print(tokenizer.decode(gen_ids[0], skip_special_tokens=True))

백지연의 소통 능력이 굉장히 뛰어나고 다양한 사람들과 인터뷰 경력이 있잖아요. 그래서 타인을 이해할 수 있는 폭이 넓지 않을까 라는 생각을 했습니다. 그리고 이 책을 읽으면서 인터뷰를 하고 다른 사람의 이야기를 듣는 법에 대해서도 깨닫게 된 것 같습니다. 인터뷰를 하면서 다른 사람들의 이야기를 들어야 한다는 것을 깨달은 것 같아 감명 깊습니다. 또한 인터뷰를 하는 방법에 대해 깨닫는 것 같았습니다.
